# Phase 4: Explainability Analysis — Project Concrete Vision
**Apex Structural Consultants × National Infrastructure Rail**

---

## Purpose of This Notebook

This notebook implements **post-hoc explainability** for the trained CNN crack-detection classifier. The goal is not just to measure *whether* the model is correct, but to understand *why* it made each decision — and critically, **what actionable changes** (data augmentation, architecture modifications, re-training strategies) should follow from those explanations.

### Why Explainability Matters Here
In structural safety applications, a model that achieves high accuracy for the **wrong reasons** is dangerous. For example:
- A model that classifies an image as *Cracked* because of a **vertical shadow** rather than an actual crack has learned a spurious correlation.
- A model that ignores fine hairline cracks because it only recognises **wide, high-contrast cracks** will systematically miss early-stage structural damage.

Explainability allows us to **audit the model's attention** and provide the client (National Infrastructure Rail) with evidence that the AI is detecting real defects — not image artefacts.

### Techniques Covered
| Technique | What it shows | Best used for |
|-----------|--------------|---------------|
| **Grad-CAM** | Class-discriminative regions via gradient-weighted feature maps | Correct & incorrect predictions |
| **Guided Backpropagation** | Fine-grained pixel-level importance | Hairline crack detection |
| **Guided Grad-CAM** | Combination: spatial + fine-grained detail | Final report collages |
| **Saliency Maps** | Raw input-gradient sensitivity | Quick sanity checks |

### Notebook Workflow
```
1. Setup & Imports
2. Load Model & Data
3. Generate Grad-CAM Heatmaps
4. Guided Backpropagation
5. Guided Grad-CAM
6. Failure Mode Analysis (False Positives & False Negatives)
7. Actionable Decision Framework
8. Report Collage Generation
```

---
> **Prerequisites:** Run this notebook only after your CNN model has been trained and saved (Phase 2). You will need the saved model file (`.pth` / `.h5` / `.keras`) and your test/validation image directory.

## 1. Setup & Imports

Install any missing packages. The core libraries needed are:
- `torch` / `torchvision` (or `tensorflow`/`keras` — adapt as needed)
- `grad-cam` — the `pytorch-grad-cam` library provides clean implementations of all techniques
- `opencv-python` — for image overlay and heatmap rendering
- `matplotlib` / `seaborn` — for plotting

> **Team Decision Point:** Confirm which framework your CNN was built in (PyTorch vs. Keras/TensorFlow) and uncomment the appropriate import block below. The Grad-CAM logic differs slightly between frameworks.

In [ ]:
# ── Install dependencies (run once) ──────────────────────────────────────────
# Uncomment as needed:
# !pip install grad-cam          # pytorch-grad-cam library
# !pip install torch torchvision
# !pip install opencv-python
# !pip install matplotlib seaborn
# !pip install Pillow
# !pip install scikit-learn
# !pip install tf-keras-vis      # If using TensorFlow/Keras instead

In [ ]:
# ── Core Imports ──────────────────────────────────────────────────────────────
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import seaborn as sns
import cv2
from PIL import Image
from pathlib import Path
from sklearn.metrics import confusion_matrix, classification_report

# ── PyTorch Imports (comment out if using TensorFlow) ─────────────────────────
import torch
import torch.nn as nn
import torchvision.transforms as transforms
from torchvision import models
from torch.utils.data import DataLoader, Dataset

# pytorch-grad-cam library — provides Grad-CAM, Guided Backprop, Guided Grad-CAM
from pytorch_grad_cam import GradCAM, GuidedBackpropReLUModel
from pytorch_grad_cam.utils.image import show_cam_on_image, preprocess_image
from pytorch_grad_cam.utils.model_targets import ClassifierOutputTarget

# ── (Alternative) TensorFlow / Keras Imports ─────────────────────────────────
# import tensorflow as tf
# from tf_keras_vis.gradcam import Gradcam
# from tf_keras_vis.utils.model_modifiers import ReplaceToLinear
# from tf_keras_vis.utils.scores import CategoricalScore

# ── Global Settings ───────────────────────────────────────────────────────────
plt.rcParams['figure.dpi'] = 150
SEED = 42
torch.manual_seed(SEED)
np.random.seed(SEED)

# Class labels — must match the order used during training
CLASS_NAMES = ['Non-Cracked', 'Cracked']   # index 0 = Non-Cracked, 1 = Cracked
POSITIVE_CLASS_IDX = 1                     # "Cracked" is the positive class

print("All imports successful.")

## 2. Configuration & Path Setup

Set all file paths here. This is the **only cell you should need to edit** when switching between different trained models or datasets.

> **Team Action:** Fill in the paths below before running the rest of the notebook.

In [ ]:
# ── Edit These Paths ──────────────────────────────────────────────────────────

MODEL_PATH       = "../models/best_model.pth"        # Path to saved model weights
VAL_IMAGE_DIR    = "../data/validation"              # Validation set (with Cracked/Non-Cracked subfolders)
MYSTERY_TEST_DIR = "../data/Mystery_Test_Set"        # Mystery test set (unlabelled)
OUTPUT_DIR       = "../outputs/phase4_explainability" # Where to save figures and CSVs
PREDICTIONS_CSV  = "../outputs/mystery_predictions.csv"  # From Phase 3

# ── Model Hyperparameters (must match training) ───────────────────────────────
IMAGE_SIZE   = (224, 224)   # (height, width) — change if you used a different size
BATCH_SIZE   = 16
DEVICE       = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ── Create output directory ───────────────────────────────────────────────────
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "gradcam"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "guided_gradcam"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "failure_modes"), exist_ok=True)
os.makedirs(os.path.join(OUTPUT_DIR, "collages"), exist_ok=True)

print(f"Device: {DEVICE}")
print(f"Output directory: {OUTPUT_DIR}")

## 3. Load Trained Model

Load the CNN architecture and restore the saved weights from Phase 2.

### What to Do Here
- **Recreate the exact architecture** used during training — if you used ResNet-18, use ResNet-18 here. If you built a custom CNN, paste its class definition below.
- Set the model to **evaluation mode** (`model.eval()`) — this disables dropout and batch normalisation in inference mode.
- Identify the **target convolutional layer** for Grad-CAM. For most architectures, this is the last convolutional block. See the guidance in the code comments below.

In [ ]:
# ── Option A: Load a pre-trained backbone (e.g. ResNet-18 with Transfer Learning) ──
def load_resnet_model(model_path, num_classes=2, device=DEVICE):
    """Load a fine-tuned ResNet-18 model."""
    model = models.resnet18(pretrained=False)       # Don't re-download weights
    model.fc = nn.Linear(model.fc.in_features, num_classes)  # Must match training head
    model.load_state_dict(torch.load(model_path, map_location=device))
    model = model.to(device)
    model.eval()
    return model

# ── Option B: Load your custom CNN ────────────────────────────────────────────
# class YourCustomCNN(nn.Module):
#     def __init__(self):
#         super().__init__()
#         # *** PASTE YOUR MODEL ARCHITECTURE HERE ***
#     def forward(self, x):
#         # *** PASTE YOUR FORWARD PASS HERE ***
#         pass
#
# def load_custom_model(model_path, device=DEVICE):
#     model = YourCustomCNN()
#     model.load_state_dict(torch.load(model_path, map_location=device))
#     model = model.to(device)
#     model.eval()
#     return model

# ── Load the model ─────────────────────────────────────────────────────────────
model = load_resnet_model(MODEL_PATH)           # ← swap for load_custom_model() if needed
print(model)

# ── Identify the target layer for Grad-CAM ────────────────────────────────────
# The target layer should be the LAST convolutional layer before the classifier.
# For ResNet-18: model.layer4[-1]  (last residual block)
# For VGG-16:    model.features[-1]
# For custom CNN: replace with your final Conv2D layer name
TARGET_LAYER = [model.layer4[-1]]   # ← UPDATE THIS FOR YOUR ARCHITECTURE

print(f"\nTarget Grad-CAM layer: {TARGET_LAYER}")

## 4. Data Loading & Preprocessing

Load the **validation set** (labelled) and the **Mystery Test Set** (unlabelled). We use the validation set for failure-mode analysis because we know the ground truth. The mystery set predictions (from Phase 3) are loaded to cross-reference with the Grad-CAM outputs.

> **Important:** Use **exactly the same preprocessing transforms** as during training. Any difference in normalisation or resize will corrupt the Grad-CAM gradients.

In [ ]:
# ── Preprocessing transforms — MUST MATCH TRAINING TRANSFORMS ─────────────────
# These values (mean/std) are the ImageNet defaults — correct if you used Transfer Learning.
# If you trained from scratch with different normalisation, update these values.
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

inference_transform = transforms.Compose([
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD)
])

# ── Custom Dataset class for labelled data (Cracked / Non-Cracked subfolders) ─
class CrackDataset(Dataset):
    """Expects directory structure: root/Cracked/*.jpg and root/Non-Cracked/*.jpg"""
    def __init__(self, root_dir, transform=None):
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.samples   = []   # list of (image_path, label)
        self.class_to_idx = {name: idx for idx, name in enumerate(CLASS_NAMES)}
        
        for class_name in CLASS_NAMES:
            class_dir = self.root_dir / class_name
            if not class_dir.exists():
                print(f"WARNING: Expected folder not found: {class_dir}")
                continue
            for img_path in class_dir.glob("*.[jJpP][pPnN][gG]*"):
                self.samples.append((img_path, self.class_to_idx[class_name]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        img_path, label = self.samples[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, label, str(img_path)

# ── Custom Dataset class for unlabelled mystery set ───────────────────────────
class UnlabelledDataset(Dataset):
    def __init__(self, root_dir, transform=None):
        self.root_dir  = Path(root_dir)
        self.transform = transform
        self.image_paths = sorted(self.root_dir.glob("*.[jJpP][pPnN][gG]*"))
    
    def __len__(self):
        return len(self.image_paths)
    
    def __getitem__(self, idx):
        img_path = self.image_paths[idx]
        image = Image.open(img_path).convert("RGB")
        if self.transform:
            image = self.transform(image)
        return image, str(img_path)

# ── Instantiate datasets ──────────────────────────────────────────────────────
val_dataset     = CrackDataset(VAL_IMAGE_DIR,    transform=inference_transform)
mystery_dataset = UnlabelledDataset(MYSTERY_TEST_DIR, transform=inference_transform)

val_loader     = DataLoader(val_dataset,     batch_size=BATCH_SIZE, shuffle=False)
mystery_loader = DataLoader(mystery_dataset, batch_size=BATCH_SIZE, shuffle=False)

print(f"Validation samples:    {len(val_dataset)}")
print(f"Mystery test samples:  {len(mystery_dataset)}")

## 5. Run Inference & Collect Predictions

Before generating explanations, we need the model's predictions and confidence scores for every image in the validation set. These are used to:
1. Identify **False Positives** (Non-Cracked predicted as Cracked) — potential spurious features
2. Identify **False Negatives** (Cracked predicted as Non-Cracked) — safety-critical misses
3. Select **confident correct** predictions for comparison

In [ ]:
# ── Run full inference pass on validation set ─────────────────────────────────
all_preds    = []
all_labels   = []
all_probs    = []
all_paths    = []

softmax = nn.Softmax(dim=1)

with torch.no_grad():
    for images, labels, paths in val_loader:
        images = images.to(DEVICE)
        outputs = model(images)
        probs   = softmax(outputs)
        preds   = torch.argmax(probs, dim=1)
        
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(labels.numpy())
        all_probs.extend(probs.cpu().numpy())
        all_paths.extend(paths)

# ── Build a results DataFrame ─────────────────────────────────────────────────
results_df = pd.DataFrame({
    'filepath':       all_paths,
    'true_label':     [CLASS_NAMES[i] for i in all_labels],
    'predicted_label':[CLASS_NAMES[i] for i in all_preds],
    'prob_non_cracked': [p[0] for p in all_probs],
    'prob_cracked':     [p[1] for p in all_probs],
    'correct':        [t == p for t, p in zip(all_labels, all_preds)]
})

# Tag each row by prediction outcome
def outcome_tag(row):
    if row['true_label'] == 'Cracked' and row['predicted_label'] == 'Cracked':
        return 'True Positive'
    elif row['true_label'] == 'Non-Cracked' and row['predicted_label'] == 'Non-Cracked':
        return 'True Negative'
    elif row['true_label'] == 'Non-Cracked' and row['predicted_label'] == 'Cracked':
        return 'False Positive'
    else:
        return 'False Negative'

results_df['outcome'] = results_df.apply(outcome_tag, axis=1)

print(results_df['outcome'].value_counts())
results_df.head(10)

## 6. Confusion Matrix & Performance Summary

Before diving into explainability, confirm overall model performance. The confusion matrix tells us **how many** failures there are; the subsequent Grad-CAM analysis tells us **why**.

> **Key metric reminder:** Recall (Sensitivity) = TP / (TP + FN). A False Negative (missed crack) is the most dangerous outcome in this application.

In [ ]:
# ── Confusion Matrix ──────────────────────────────────────────────────────────
cm = confusion_matrix(all_labels, all_preds)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm, annot=True, fmt='d', cmap='Blues',
    xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES,
    linewidths=0.5, ax=ax
)
ax.set_title('Validation Set — Confusion Matrix', fontsize=14, fontweight='bold')
ax.set_ylabel('True Label', fontsize=12)
ax.set_xlabel('Predicted Label', fontsize=12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, 'confusion_matrix.png'), dpi=150)
plt.show()

# ── Classification Report ─────────────────────────────────────────────────────
print("\nClassification Report:")
print(classification_report(all_labels, all_preds, target_names=CLASS_NAMES))

tn, fp, fn, tp = cm.ravel()
recall    = tp / (tp + fn) if (tp + fn) > 0 else 0
precision = tp / (tp + fp) if (tp + fp) > 0 else 0
print(f"\nCrack Detection Recall (Sensitivity): {recall:.4f}")
print(f"Crack Detection Precision:            {precision:.4f}")
print(f"False Negatives (missed cracks):      {fn}  ← SAFETY CRITICAL")
print(f"False Positives (false alarms):       {fp}")

## 7. Grad-CAM — Class Activation Mapping

### What Grad-CAM Does
Grad-CAM computes the **gradient of the target class score** with respect to the feature maps of a chosen convolutional layer. Regions with large positive gradients are highlighted with warm colours (red/yellow), indicating the areas most responsible for the classification decision.

### Reading the Heatmap
| Colour | Meaning |
|--------|----------|
| 🔴 Red / Yellow | High activation — model focused here |
| 🔵 Blue / Dark | Low activation — model ignored this region |

### What to Look For
- **Good:** Heatmap concentrated on the crack path itself
- **Suspicious:** Heatmap concentrated on edges, borders, shadows, or background texture
- **Dangerous:** Heatmap concentrated away from any visible defect on a False Negative

In [ ]:
# ── Helper: load raw image as float32 numpy array in [0, 1] ───────────────────
def load_rgb_float(img_path, size=IMAGE_SIZE):
    """Load image as float32 numpy array (H, W, 3) normalised to [0, 1]."""
    img = Image.open(img_path).convert("RGB").resize((size[1], size[0]))
    return np.array(img, dtype=np.float32) / 255.0

# ── Helper: prepare a single image tensor for the model ───────────────────────
def prepare_tensor(img_path, transform=inference_transform, device=DEVICE):
    img   = Image.open(img_path).convert("RGB")
    tensor = transform(img).unsqueeze(0).to(device)   # shape: (1, C, H, W)
    return tensor

# ── Core Grad-CAM generation function ────────────────────────────────────────
def generate_gradcam(model, target_layers, img_path, target_class_idx=None):
    """
    Generate a Grad-CAM heatmap overlaid on the original image.
    
    Parameters:
    -----------
    model            : trained PyTorch model in eval mode
    target_layers    : list of layers to hook (e.g. [model.layer4[-1]])
    img_path         : path to the input image
    target_class_idx : class to explain (None = argmax / predicted class)
                       Set to POSITIVE_CLASS_IDX (1) to always explain "Cracked"
    
    Returns:
    --------
    cam_image : numpy array (H, W, 3) — heatmap overlaid on original image
    grayscale_cam : numpy array (H, W) — raw heatmap values in [0, 1]
    """
    rgb_img    = load_rgb_float(img_path)
    input_tensor = prepare_tensor(img_path)
    
    targets = [ClassifierOutputTarget(target_class_idx)] if target_class_idx is not None else None
    
    with GradCAM(model=model, target_layers=target_layers) as cam:
        grayscale_cam = cam(input_tensor=input_tensor, targets=targets)
        grayscale_cam = grayscale_cam[0]   # shape: (H, W)
    
    cam_image = show_cam_on_image(rgb_img, grayscale_cam, use_rgb=True)
    return cam_image, grayscale_cam

print("Grad-CAM helper functions defined.")

In [ ]:
# ── Generate and display Grad-CAM for a sample of images ─────────────────────
# Select a stratified sample: 2 of each outcome type for inspection
N_SAMPLES_PER_CATEGORY = 2   # ← increase if you want a richer analysis

sample_rows = pd.concat([
    results_df[results_df['outcome'] == cat].nlargest(N_SAMPLES_PER_CATEGORY, 'prob_cracked')
    for cat in ['True Positive', 'True Negative', 'False Positive', 'False Negative']
]).reset_index(drop=True)

n_cols = 3   # original | Grad-CAM | label info
n_rows = len(sample_rows)

fig, axes = plt.subplots(n_rows, n_cols, figsize=(14, n_rows * 4))
fig.suptitle('Grad-CAM Analysis — Stratified Sample', fontsize=16, fontweight='bold', y=1.01)

for i, row in sample_rows.iterrows():
    img_path = row['filepath']
    
    # Original image
    orig_img = load_rgb_float(img_path)
    axes[i, 0].imshow(orig_img)
    axes[i, 0].set_title('Original', fontsize=9)
    axes[i, 0].axis('off')
    
    # Grad-CAM for "Cracked" class (always explain the positive class)
    cam_img, _ = generate_gradcam(model, TARGET_LAYER, img_path, target_class_idx=POSITIVE_CLASS_IDX)
    axes[i, 1].imshow(cam_img)
    axes[i, 1].set_title(f'Grad-CAM (→ Cracked class)', fontsize=9)
    axes[i, 1].axis('off')
    
    # Label panel
    outcome_colour = {
        'True Positive': 'green', 'True Negative': 'blue',
        'False Positive': 'darkorange', 'False Negative': 'red'
    }
    colour = outcome_colour[row['outcome']]
    label_text = (
        f"Outcome: {row['outcome']}\n"
        f"True:    {row['true_label']}\n"
        f"Pred:    {row['predicted_label']}\n"
        f"P(crack)={row['prob_cracked']:.3f}"
    )
    axes[i, 2].text(0.5, 0.5, label_text, transform=axes[i, 2].transAxes,
                    ha='center', va='center', fontsize=10,
                    color=colour, fontweight='bold',
                    bbox=dict(boxstyle='round', facecolor='wheat', alpha=0.5))
    axes[i, 2].axis('off')

plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'gradcam', 'gradcam_stratified_sample.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Saved: {save_path}")

## 8. Guided Backpropagation & Guided Grad-CAM

### Why Go Beyond Basic Grad-CAM?
Grad-CAM produces **coarse, blob-like** heatmaps because it operates on the low-resolution feature maps of a deep layer. For detecting **hairline cracks**, this resolution may be too low to confirm whether the model is tracking the crack path precisely.

**Guided Backpropagation** produces pixel-level saliency by only propagating *positive* gradients back through ReLU layers. It reveals fine-grained structure but can be noisy.

**Guided Grad-CAM** = element-wise multiplication of Grad-CAM × Guided Backprop. This gives both **spatial localisation AND fine detail** — the best of both worlds.

> **Use Guided Grad-CAM for your report collage** — it is the most visually convincing technique for client presentation.

In [ ]:
# ── Guided Backpropagation ────────────────────────────────────────────────────
def generate_guided_backprop(model, img_path, target_class_idx=POSITIVE_CLASS_IDX, device=DEVICE):
    """
    Generate a Guided Backpropagation saliency map.
    Returns a grayscale numpy array (H, W) normalised to [0, 1].
    """
    gb_model = GuidedBackpropReLUModel(model=model, device=device)
    input_tensor = prepare_tensor(img_path)
    
    gb_output = gb_model(input_tensor, target_category=target_class_idx)
    
    # Convert to grayscale by taking max across colour channels
    gb_gray = np.max(np.abs(gb_output), axis=2)
    gb_gray = (gb_gray - gb_gray.min()) / (gb_gray.max() - gb_gray.min() + 1e-8)
    return gb_gray

# ── Guided Grad-CAM: multiply Grad-CAM heatmap × Guided Backprop ──────────────
def generate_guided_gradcam(model, target_layers, img_path, target_class_idx=POSITIVE_CLASS_IDX, device=DEVICE):
    """
    Generate Guided Grad-CAM = Grad-CAM ⊙ Guided Backprop.
    Returns:
      guided_gradcam : float32 numpy array (H, W), normalised
      cam_image      : RGB overlay (H, W, 3)
    """
    _, grayscale_cam  = generate_gradcam(model, target_layers, img_path, target_class_idx)
    gb_gray           = generate_guided_backprop(model, img_path, target_class_idx, device)
    
    guided_gc = grayscale_cam * gb_gray
    guided_gc = (guided_gc - guided_gc.min()) / (guided_gc.max() - guided_gc.min() + 1e-8)
    
    rgb_img   = load_rgb_float(img_path)
    cam_image = show_cam_on_image(rgb_img, guided_gc, use_rgb=True)
    return guided_gc, cam_image

print("Guided Backprop & Guided Grad-CAM functions defined.")

In [ ]:
# ── Compare all three techniques side-by-side for selected images ─────────────
# Select a handful of interesting cases for the detailed comparison
COMPARE_CASES = sample_rows.head(4)['filepath'].tolist()   # ← swap for specific paths if desired

fig, axes = plt.subplots(len(COMPARE_CASES), 4, figsize=(18, len(COMPARE_CASES) * 4))
col_titles = ['Original', 'Grad-CAM', 'Guided Backprop', 'Guided Grad-CAM']
for ax, title in zip(axes[0], col_titles):
    ax.set_title(title, fontsize=12, fontweight='bold')

for i, img_path in enumerate(COMPARE_CASES):
    orig          = load_rgb_float(img_path)
    cam_img, _    = generate_gradcam(model, TARGET_LAYER, img_path, POSITIVE_CLASS_IDX)
    gb_map         = generate_guided_backprop(model, img_path)
    _, ggc_img    = generate_guided_gradcam(model, TARGET_LAYER, img_path)
    
    for ax, vis in zip(axes[i], [orig, cam_img, gb_map, ggc_img]):
        ax.imshow(vis, cmap='hot' if vis.ndim == 2 else None)
        ax.axis('off')

plt.suptitle('Technique Comparison: Grad-CAM vs Guided Backprop vs Guided Grad-CAM',
             fontsize=14, fontweight='bold', y=1.01)
plt.tight_layout()
save_path = os.path.join(OUTPUT_DIR, 'guided_gradcam', 'technique_comparison.png')
plt.savefig(save_path, bbox_inches='tight', dpi=150)
plt.show()
print(f"Saved: {save_path}")

## 9. Failure Mode Analysis — Deep Dive

This is the **most important analytical section** of the notebook. We isolate **False Positives** and **False Negatives** and examine *exactly* what features triggered or suppressed the crack classification.

### Taxonomy of Common Failure Modes
Use the Grad-CAM evidence to classify each failure into one of these categories:

| Failure Mode | Description | Likely Fix |
|---|---|---|
| **Shadow confusion** | Linear shadows mistaken for cracks | Augment with simulated shadows; brightness/contrast jitter |
| **Texture/stain confusion** | Water stains or rust patches trigger crack class | Hard-negative mining; class-weighted loss |
| **Joint line confusion** | Construction joints (normal) flagged as cracks | Include joint images in Non-Cracked training data |
| **Low-contrast miss** | Hairline cracks missed because too subtle | Contrast-enhancement preprocessing; higher-res input |
| **Domain shift miss** | Model trained on Walls fails on Deck/Pavement texture | Multi-domain training; domain adversarial training |
| **Edge artefact** | Image borders/corners activate the crack class | Crop augmentation; padding removal |

> **Action:** After generating the visualisations below, annotate each case with one of the above failure modes. The aggregated tally directly informs the remediation plan in Section 10.

In [ ]:
# ── Extract False Positives and False Negatives ───────────────────────────────
false_positives = results_df[results_df['outcome'] == 'False Positive'].copy()
false_negatives = results_df[results_df['outcome'] == 'False Negative'].copy()

# Sort by model confidence to surface the "most wrong" examples
# FP: model was most confident (highest prob_cracked) but was wrong
# FN: model was most confident (lowest prob_cracked) it was non-cracked, but was wrong
false_positives = false_positives.sort_values('prob_cracked', ascending=False)
false_negatives = false_negatives.sort_values('prob_cracked', ascending=True)

print(f"Total False Positives: {len(false_positives)}")
print(f"Total False Negatives: {len(false_negatives)}")

# Limit to top N for visualisation
N_FAILURE_CASES = 6
top_fps = false_positives.head(N_FAILURE_CASES)
top_fns = false_negatives.head(N_FAILURE_CASES)

print(f"\nDisplaying top {N_FAILURE_CASES} most-confident False Positives and False Negatives.")

In [ ]:
# ── False Positive Deep-Dive: what made the model think it saw a crack? ───────
def plot_failure_mode_analysis(failure_df, failure_type, output_subdir):
    """
    For each failed image: show original + Grad-CAM + annotation placeholder.
    """
    n = len(failure_df)
    if n == 0:
        print(f"No {failure_type} cases found.")
        return
    
    fig, axes = plt.subplots(n, 3, figsize=(14, n * 4.5))
    if n == 1:
        axes = axes[np.newaxis, :]  # ensure 2D indexing
    
    title_colour = 'darkorange' if failure_type == 'False Positive' else 'red'
    fig.suptitle(
        f'{failure_type} Analysis — Top {n} Most-Confident Errors',
        fontsize=14, fontweight='bold', color=title_colour
    )
    
    for i, (_, row) in enumerate(failure_df.iterrows()):
        img_path = row['filepath']
        
        # Original
        axes[i, 0].imshow(load_rgb_float(img_path))
        axes[i, 0].set_title(f'Original\n{Path(img_path).name}', fontsize=8)
        axes[i, 0].axis('off')
        
        # Grad-CAM
        cam_img, _ = generate_gradcam(model, TARGET_LAYER, img_path, POSITIVE_CLASS_IDX)
        axes[i, 1].imshow(cam_img)
        axes[i, 1].set_title(f'Grad-CAM (Cracked class)\nP(crack)={row["prob_cracked"]:.3f}', fontsize=8)
        axes[i, 1].axis('off')
        
        # Manual annotation placeholder
        note = (
            f"TRUE:  {row['true_label']}\n"
            f"PRED:  {row['predicted_label']}\n\n"
            "--- ANNOTATE BELOW ---\n"
            "Failure mode:\n"
            "[ ] Shadow confusion\n"
            "[ ] Texture/stain\n"
            "[ ] Joint line\n"
            "[ ] Low-contrast miss\n"
            "[ ] Domain shift\n"
            "[ ] Edge artefact\n"
            "[ ] Other: _______"
        )
        axes[i, 2].text(0.05, 0.95, note, transform=axes[i, 2].transAxes,
                        ha='left', va='top', fontsize=8, family='monospace',
                        bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))
        axes[i, 2].axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    save_path = os.path.join(OUTPUT_DIR, 'failure_modes', f'{failure_type.lower().replace(" ", "_")}_analysis.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved: {save_path}")

# ── Run for both failure types ─────────────────────────────────────────────────
plot_failure_mode_analysis(top_fps, 'False Positive', 'false_positives')
plot_failure_mode_analysis(top_fns, 'False Negative', 'false_negatives')

## 10. Actionable Decision Framework

After reviewing the Grad-CAM failure analysis above, populate the table below. This provides the **evidence base** for any recommended changes to the training pipeline.

### How to Use This Section

1. For each failure image, decide which failure mode it falls into (see the taxonomy in Section 9).
2. Tally them up in the `failure_counts` dictionary below.
3. Run the cell — it will generate a bar chart and a prioritised recommendation table.
4. Use this output directly in your report's **Recommendation** section.

In [ ]:
# ── FILL THIS IN based on your manual annotation of the failure images above ──
# These are placeholder counts — replace with your actual findings.
failure_counts = {
    'Shadow confusion':    0,   # ← update after reviewing Grad-CAM outputs
    'Texture/stain':       0,
    'Joint line':          0,
    'Low-contrast miss':   0,
    'Domain shift':        0,
    'Edge artefact':       0,
    'Other':               0,
}

# Corresponding recommended fixes for each failure mode
remediation_map = {
    'Shadow confusion':  [
        "Add random brightness/contrast augmentation (torchvision.transforms.ColorJitter)",
        "Add random shadow simulation augmentation (albumentations.RandomShadow)",
        "Review and potentially remove images with severe shadow artefacts from training data"
    ],
    'Texture/stain': [
        "Apply hard-negative mining: collect FP images, add to Non-Cracked training fold",
        "Use focal loss or class-weighted cross-entropy to penalise FP errors more",
        "Consider adding a post-processing confidence threshold above 0.5 for the crack class"
    ],
    'Joint line': [
        "Explicitly add joint-line images to Non-Cracked training set",
        "Collect or synthesise images of construction joints for data augmentation"
    ],
    'Low-contrast miss': [
        "Apply CLAHE (Contrast Limited Adaptive Histogram Equalisation) as a preprocessing step",
        "Increase input image resolution (e.g., 256×256 or 299×299)",
        "Try a model with higher-resolution feature maps (e.g., EfficientNet-B3)"
    ],
    'Domain shift': [
        "Train on combined Wall + Deck + Pavement data (if available)",
        "Apply domain adversarial training to learn domain-invariant crack features",
        "Recommend separate specialist models per asset type to the client",
        "Apply test-time augmentation (TTA) on the mystery set predictions"
    ],
    'Edge artefact': [
        "Add random crop augmentation during training to reduce reliance on image borders",
        "Apply centre-crop or padding removal during preprocessing"
    ],
    'Other': [
        "Document and describe the specific artefact; assess on a case-by-case basis"
    ]
}

# ── Visualise failure mode distribution ──────────────────────────────────────
fc_df = pd.DataFrame.from_dict(failure_counts, orient='index', columns=['Count'])
fc_df = fc_df[fc_df['Count'] > 0].sort_values('Count', ascending=False)

if len(fc_df) == 0:
    print("No failure modes recorded yet. Fill in 'failure_counts' above after reviewing the images.")
else:
    fig, ax = plt.subplots(figsize=(9, 4))
    bars = ax.barh(fc_df.index, fc_df['Count'], color='tomato', edgecolor='black')
    ax.bar_label(bars, padding=3, fontsize=10)
    ax.set_xlabel('Number of Errors', fontsize=12)
    ax.set_title('Failure Mode Frequency — Grad-CAM Evidence', fontsize=13, fontweight='bold')
    ax.invert_yaxis()
    plt.tight_layout()
    save_path = os.path.join(OUTPUT_DIR, 'failure_mode_distribution.png')
    plt.savefig(save_path, dpi=150)
    plt.show()
    print(f"Saved: {save_path}")

In [ ]:
# ── Generate prioritised remediation table ────────────────────────────────────
print("=" * 70)
print("PRIORITISED REMEDIATION RECOMMENDATIONS")
print("Based on Grad-CAM failure mode analysis")
print("=" * 70)

active_modes = {k: v for k, v in failure_counts.items() if v > 0}

if not active_modes:
    print("\nNo active failure modes recorded. Populate 'failure_counts' to generate recommendations.")
else:
    sorted_modes = sorted(active_modes.items(), key=lambda x: -x[1])
    for rank, (mode, count) in enumerate(sorted_modes, 1):
        print(f"\n{'─' * 60}")
        print(f"PRIORITY {rank}: {mode}  [{count} case(s)]")
        print(f"{'─' * 60}")
        for action in remediation_map.get(mode, ['No specific action defined.']):
            print(f"  • {action}")

print("\n" + "=" * 70)

## 11. Mystery Test Set — Explainability on Unlabelled Predictions

Apply Grad-CAM to a sample of **high-confidence** and **low-confidence** mystery set predictions. This serves two purposes:

1. **Evidence for the report:** Provide visual proof that the model attends to crack-like features even on unseen asset types.
2. **Domain shift investigation:** Compare where the model looks on *Wall* images vs. potentially *Pavement* or *Deck* images in the mystery set.

> **Note:** Because the mystery set has no labels, we cannot compute accuracy here. We are purely using explainability to *audit the model's behaviour* on out-of-distribution data.

In [ ]:
# ── Load Phase 3 mystery set predictions (produced in Phase 3 notebook) ───────
mystery_preds_df = pd.read_csv(PREDICTIONS_CSV)
print(f"Loaded {len(mystery_preds_df)} mystery predictions.")
print(mystery_preds_df.head())
print("\nPrediction distribution:")
print(mystery_preds_df['Prediction'].value_counts())

In [ ]:
# ── Select representative mystery samples for Grad-CAM ────────────────────────
# We need to load raw images; reconstruct filepaths from filenames
mystery_preds_df['filepath'] = mystery_preds_df['Filename'].apply(
    lambda f: os.path.join(MYSTERY_TEST_DIR, f)
)

# Rerun inference to get probability scores (CSV only has binary labels)
mystery_probs = []
mystery_filenames = []

with torch.no_grad():
    for images, paths in mystery_loader:
        images  = images.to(DEVICE)
        outputs = model(images)
        probs   = softmax(outputs)
        mystery_probs.extend(probs.cpu().numpy())
        mystery_filenames.extend([Path(p).name for p in paths])

mystery_detail_df = pd.DataFrame({
    'Filename':        mystery_filenames,
    'prob_non_cracked': [p[0] for p in mystery_probs],
    'prob_cracked':    [p[1] for p in mystery_probs],
})
mystery_detail_df = mystery_detail_df.merge(mystery_preds_df[['Filename', 'Prediction', 'filepath']], on='Filename')

# Select high-confidence cracked and non-cracked samples
N_MYSTERY_SAMPLES = 4
high_conf_cracked    = mystery_detail_df[mystery_detail_df['Prediction'] == 'Cracked'].nlargest(N_MYSTERY_SAMPLES, 'prob_cracked')
high_conf_noncracked = mystery_detail_df[mystery_detail_df['Prediction'] == 'Non-Cracked'].nlargest(N_MYSTERY_SAMPLES, 'prob_non_cracked')
low_conf             = mystery_detail_df.assign(
    confidence=mystery_detail_df[['prob_non_cracked', 'prob_cracked']].max(axis=1)
).nsmallest(N_MYSTERY_SAMPLES, 'confidence')

print(f"High-confidence Cracked samples:     {len(high_conf_cracked)}")
print(f"High-confidence Non-Cracked samples: {len(high_conf_noncracked)}")
print(f"Low-confidence (uncertain) samples:  {len(low_conf)}")

In [ ]:
# ── Plot Grad-CAM for mystery samples ─────────────────────────────────────────
def plot_mystery_gradcam(sample_df, subset_label):
    n = len(sample_df)
    if n == 0:
        print(f"No samples to show for: {subset_label}")
        return
    
    fig, axes = plt.subplots(n, 2, figsize=(10, n * 4))
    if n == 1:
        axes = axes[np.newaxis, :]
    
    fig.suptitle(f'Mystery Test Set — Grad-CAM: {subset_label}',
                 fontsize=13, fontweight='bold')
    
    for i, (_, row) in enumerate(sample_df.iterrows()):
        axes[i, 0].imshow(load_rgb_float(row['filepath']))
        axes[i, 0].set_title(f"{row['Filename']}\nPred: {row['Prediction']} | P(crack)={row['prob_cracked']:.3f}", fontsize=8)
        axes[i, 0].axis('off')
        
        cam_img, _ = generate_gradcam(model, TARGET_LAYER, row['filepath'], POSITIVE_CLASS_IDX)
        axes[i, 1].imshow(cam_img)
        axes[i, 1].set_title('Grad-CAM (Cracked class)', fontsize=8)
        axes[i, 1].axis('off')
    
    plt.tight_layout(rect=[0, 0, 1, 0.97])
    fname = subset_label.lower().replace(' ', '_').replace('(', '').replace(')', '')
    save_path = os.path.join(OUTPUT_DIR, 'gradcam', f'mystery_{fname}.png')
    plt.savefig(save_path, bbox_inches='tight', dpi=150)
    plt.show()
    print(f"Saved: {save_path}")

plot_mystery_gradcam(high_conf_cracked,    "High-Confidence Cracked Predictions")
plot_mystery_gradcam(high_conf_noncracked, "High-Confidence Non-Cracked Predictions")
plot_mystery_gradcam(low_conf,             "Low-Confidence (Uncertain) Predictions")

## 12. Report Collage Generation

The brief requires a **collage** showing examples of:
- Images where the model correctly focused on a crack ✅
- Images where the model focused on a spurious feature (shadow, stain, leaf) ⚠️

This cell generates a publication-quality collage grid suitable for direct inclusion in the Phase 4 report section.

> **Team Action:** Select specific images for each panel using the `collage_cases` list below. Ideally include at least one example of each type of spurious feature found in Section 9.

In [ ]:
# ── CONFIGURE: Manually specify images for the report collage ─────────────────
# Replace the placeholder paths with specific images that best illustrate each case.
# Each entry: (image_path, panel_label, annotation_text)

collage_cases = [
    # Format: (filepath, panel_title, short_annotation)
    # CORRECT DETECTIONS — model attended to actual crack
    # ("path/to/image1.jpg", "Correct: Wall Crack",     "Model focuses\non crack path ✓"),
    # ("path/to/image2.jpg", "Correct: Deck Crack",     "Attention tracks\nhairline crack ✓"),
    
    # SPURIOUS DETECTIONS — model attended to non-crack feature
    # ("path/to/image3.jpg", "FP: Vertical Shadow",    "Shadow mistaken\nfor crack ✗"),
    # ("path/to/image4.jpg", "FP: Water Stain",        "Stain pattern\ntriggers crack class ✗"),
    
    # FALSE NEGATIVES — model missed the crack
    # ("path/to/image5.jpg", "FN: Hairline Crack",     "Low contrast crack\nnot detected ✗"),
]

# ── If no cases are manually specified, fall back to auto-selected examples ───
if not collage_cases:
    print("No manual collage cases specified. Auto-selecting from results_df...")
    auto_tp = results_df[results_df['outcome'] == 'True Positive'].nlargest(2, 'prob_cracked')
    auto_fp = results_df[results_df['outcome'] == 'False Positive'].nlargest(2, 'prob_cracked')
    auto_fn = results_df[results_df['outcome'] == 'False Negative'].nsmallest(2, 'prob_cracked')
    
    for _, row in auto_tp.iterrows():
        collage_cases.append((row['filepath'], f"TP: Correct Crack", f"P(crack)={row['prob_cracked']:.2f}"))
    for _, row in auto_fp.iterrows():
        collage_cases.append((row['filepath'], f"FP: False Alarm", f"P(crack)={row['prob_cracked']:.2f}"))
    for _, row in auto_fn.iterrows():
        collage_cases.append((row['filepath'], f"FN: Missed Crack", f"P(crack)={row['prob_cracked']:.2f}"))

print(f"Collage will include {len(collage_cases)} panels.")

In [ ]:
# ── Render the Report Collage ─────────────────────────────────────────────────
n_panels = len(collage_cases)
n_cols   = min(4, n_panels)
n_rows   = -(-n_panels // n_cols)   # ceiling division → rows needed

fig = plt.figure(figsize=(n_cols * 5, n_rows * 5.5))
fig.patch.set_facecolor('#1a1a2e')

for idx, (img_path, title, annotation) in enumerate(collage_cases):
    ax_orig = fig.add_subplot(n_rows * 2, n_cols, idx + 1)
    ax_cam  = fig.add_subplot(n_rows * 2, n_cols, idx + 1 + n_cols * n_rows)
    
    orig = load_rgb_float(img_path)
    cam_img, _ = generate_gradcam(model, TARGET_LAYER, img_path, POSITIVE_CLASS_IDX)
    
    ax_orig.imshow(orig)
    ax_orig.set_title(title, color='white', fontsize=9, fontweight='bold', pad=4)
    ax_orig.axis('off')
    
    ax_cam.imshow(cam_img)
    ax_cam.set_title(annotation, color='#ffd700', fontsize=8, pad=4)
    ax_cam.axis('off')

plt.suptitle(
    'Project Concrete Vision — Grad-CAM Explainability Collage\n'
    'Apex Structural Consultants × National Infrastructure Rail',
    color='white', fontsize=13, fontweight='bold', y=1.01
)
plt.tight_layout()

collage_path = os.path.join(OUTPUT_DIR, 'collages', 'report_collage_gradcam.png')
plt.savefig(collage_path, bbox_inches='tight', dpi=200, facecolor=fig.get_facecolor())
plt.show()
print(f"Report collage saved: {collage_path}")

## 13. Summary & Recommendations Export

Export a machine-readable summary of all explainability findings. This can be referenced or appended to in your written report.

In [ ]:
# ── Export full validation results with outcome tags ──────────────────────────
results_path = os.path.join(OUTPUT_DIR, 'validation_results_with_outcomes.csv')
results_df.to_csv(results_path, index=False)
print(f"Saved full results: {results_path}")

# ── Export failure mode tally ─────────────────────────────────────────────────
fm_df = pd.DataFrame.from_dict(failure_counts, orient='index', columns=['Count'])
fm_df.index.name = 'Failure_Mode'
fm_path = os.path.join(OUTPUT_DIR, 'failure_mode_tally.csv')
fm_df.to_csv(fm_path)
print(f"Saved failure mode tally: {fm_path}")

# ── Print final summary ───────────────────────────────────────────────────────
tn, fp, fn, tp = confusion_matrix(all_labels, all_preds).ravel()
total_errors = fp + fn

print("\n" + "=" * 60)
print("PHASE 4 EXPLAINABILITY SUMMARY")
print("=" * 60)
print(f"  Total validation errors:    {total_errors}")
print(f"  ├─ False Positives (FP):    {fp}  (false alarms)")
print(f"  └─ False Negatives (FN):    {fn}  ← SAFETY CRITICAL")
print(f"\n  Recall (sensitivity):       {tp/(tp+fn):.4f}")
print(f"  Precision:                  {tp/(tp+fp):.4f}")
print(f"\n  Outputs saved to:           {OUTPUT_DIR}")
print("=" * 60)

## 14. Report Write-Up Guidance

Use the outputs from this notebook to populate **Section 4 (Explainability)** of the group report. Below is a suggested structure.

---

### 4.1 Technique Justification
Explain *why* Grad-CAM was chosen over simpler methods. Key points:
- Grad-CAM is class-discriminative (unlike basic saliency maps, it targets a *specific* output class)
- It does not require model re-training or architectural changes
- It is interpretable to non-ML engineers (heatmap overlaid on the original image)
- Guided Grad-CAM provides higher resolution for thin/hairline cracks

### 4.2 Evidence of Correct Detection
Include the **True Positive** panels from the report collage. Describe what structural feature the heatmap highlights and why this constitutes valid crack detection.

### 4.3 Evidence of Spurious Detection (False Positives)
Include the **False Positive** panels. For each:
- Identify the visual artefact the model responded to (shadow, stain, joint, etc.)
- Explain *why* this feature might visually resemble a crack to a CNN (linear patterns, high-frequency edges)
- Cross-reference to the recommended fix from the remediation table

### 4.4 Evidence of Missed Cracks (False Negatives)
Include the **False Negative** panels. For each:
- Describe where the model was *not* attending (i.e., it was not looking at the crack)
- Explain the likely cause (low contrast, domain mismatch, texture similarity to background)
- This directly supports the domain shift discussion in Phase 3

### 4.5 Limitations of Grad-CAM
Be critical of the technique itself:
- Grad-CAM resolution is limited by the spatial resolution of the target layer
- It explains *what* the model focuses on, but not *why* that feature was learned
- Multiple explanations can look plausible — Grad-CAM cannot prove the model is generalising correctly

### 4.6 Recommendation: One Model or Many?
Based on your Grad-CAM analysis, synthesise a recommendation. Key evidence points to consider:
- Do the heatmaps look qualitatively different for Wall vs. Deck images?
- Is domain shift visibly affecting where the model attends on mystery set images?
- How many of the FN failures appear to be domain-shift errors?

---

> **Final note:** All figures generated by this notebook are saved to `outputs/phase4_explainability/`. Reference them directly in your LaTeX/Word report rather than re-exporting manually.